In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from multimodal_lancedb import MusicDatabase
from utils import (
    EmbeddingProcessor,
    Ranker,
    MusicSearchSystem
)
from IPython.display import Audio, display
import pandas as pd

In [2]:
# Initialize the system
search_system = MusicSearchSystem(db_path="./.lancedb_2", music_dir="music")

In [3]:
import lancedb
db = lancedb.connect("./.lancedb_2")
table_audio = db.open_table("music_audio")
audio_embedding_df = table_audio.to_pandas()

audio_embedding_df

,song_name,song_path,audio_vector
0,My Rhapsody Sounds - Short Version A,music/Assaf Ayalon - My Rhapsody Sounds - Shor...,"[-0.029478367, 0.009709472, 0.05185532, 0.0254..."
1,Laid Back - Short Version A,music/The Mind Sweepers - Laid Back - Short Ve...,"[-0.033787563, 0.040226605, 0.004442984, 0.081..."
2,Far Taj,music/ZISO - Far Taj.mp3,"[-0.018721217, 0.035051364, 0.05141652, 0.0356..."
3,The Stones - Short Version,music/Wild Tulip - The Stones - Short Version.mp3,"[-0.008583562, 0.020294761, 0.0050604204, 0.03..."
4,Fixed - Short Version B,music/Swirling Ship - Fixed - Short Version B.mp3,"[0.0031058518, 0.020443501, 0.0060475464, -0.0..."
...,...,...,...
195,Orchestral News Intro,music/Tomasz_Redman - Orchestral News Intro.mp3,"[0.010141604, -0.017981885, 0.014272054, -0.03..."
196,Upbeat Happy Fun Logo,music/puremusic - Upbeat Happy Fun Logo.mp3,"[-0.031311695, -0.03104818, 0.022266045, 0.022..."
197,Happy Birthday In Paris,music/Music-Ideas - Happy Birthday In Paris.mp3,"[-0.049790498, -0.03626891, 0.059174698, -0.00..."
198,Funny Game Loop,music/honey_lemon - Funny Game Loop.wav,"[-0.045391183, -0.033096816, 0.025908915, -0.0..."


In [4]:
table_text = db.open_table("music_text")
text_embedding_df = table_text.to_pandas()

text_embedding_df

,source,song_name,artist,mood,video_theme,genre,instrument,other_tags,bpm,lmm_description,combined_info,text_vector
0,Artlist,My Rhapsody Sounds - Short Version A,Assaf Ayalon,"Uplifting, Happy, Carefree, Love, Playful","Business, Food, Education, Lifestyle, Urban","Cinematic, Acoustic, Pop, Folk, Children, Corp...","Acoustic Guitar, Keys",,145.0,A positive and uplifting acoustic folk track w...,"Moods: Uplifting, Happy, Carefree, Love, Playf...","[0.00016941165, -0.011131651, -0.004014101, -0..."
1,Artlist,Laid Back - Short Version A,The Mind Sweepers,"Powerful, Serious, Angry","Road Trip, Sport & Fitness, Fashion, Industry",Rock,"Electric, Guitar, Acoustic Drums",,78.0,This is a powerful and energetic rock music tr...,"Moods: Powerful, Serious, Angry. Video Themes:...","[-0.0041819224, -0.01964485, -0.021090291, -0...."
2,Artlist,Far Taj,ZISO,"Uplifting, Powerful, Carefree, Groovy","Travel, Shorts","World, Electronic, Hip Hop","Ethnic, Electronic Drums, Bass",,96.0,A traditional Indian Bhangra track with modern...,"Moods: Uplifting, Powerful, Carefree, Groovy. ...","[-0.011723319, -0.008885955, 0.0040757894, -0...."
3,Artlist,The Stones - Short Version,Wild Tulip,"Love, Serious, Dramatic, Sad, Hopeful","Time-Lapse, Documentary, Road Trip, Medical, L...",Cinematic,Piano,,69.0,This piece is a solo piano instrumental with a...,"Moods: Love, Serious, Dramatic, Sad, Hopeful. ...","[0.0012076573, -0.0034555339, -0.0020917628, -..."
4,Artlist,Fixed - Short Version B,Swirling Ship,"Serious, Dramatic, Scary, Dark","Time-Lapse, Drone Shots, Nature, Slow Motion","Ambient, Country, Cinematic","Electric Guitar, Synth, Electronic Drums, Pads",,121.0,"The music is mysterious and dramatic, featurin...","Moods: Serious, Dramatic, Scary, Dark. Video T...","[-0.0021377725, -0.012590199, -0.013713269, -0..."
...,...,...,...,...,...,...,...,...,...,...,...,...
195,envato,Orchestral News Intro,Tomasz_Redman,"energetic, epic, powerful, solemn, uplifting","announcement, broadcast news, broadcasting, bu...",corporate,strings,global,125.0,This is a dynamic and uplifting music track th...,"Moods: energetic, epic, powerful, solemn, upli...","[-0.006809455, -0.016746698, -0.016968248, -0...."
196,envato,Upbeat Happy Fun Logo,puremusic,"bouncy, bright, catchy, cheerful, energetic, f...","commercial, happy logo, intro, kids, logo, sum...","acoustic, children","claps, ukulele","melody, youth",NaN,"A positive, upbeat, cheerful, and happy acoust...","Moods: bouncy, bright, catchy, cheerful, energ...","[0.002132031, -0.005020746, -0.0029374287, -0...."
197,envato,Happy Birthday In Paris,Music-Ideas,"cheerful, funny, happy, lively, playful, upbeat","ads, advertising, birthday, broadway, casino, ...","bigband, jazz, retro","accordion, piano, trumpets","france, french, paris",120.0,A fun and lively Latin track featuring a varie...,"Moods: cheerful, funny, happy, lively, playful...","[-0.012678004, -0.011919039, -0.00089095806, -..."
198,envato,Funny Game Loop,honey_lemon,"comical, fun, funny, laugh, smile, soft","cartoon, comedy, comic, kids, short, summer, tv","acoustic, children, folk, jazz",bells,loop,170.0,"A casual, jazzy, swing music with vibraphone, ...","Moods: comical, fun, funny, laugh, smile, soft...","[-0.012116052, -0.01674075, 0.010771282, -0.02..."


In [86]:
# Search the music from query
query = '''
I’m editing a fashion-themed video and need a stylish house music track that fits the mood of a modern runway or lookbook showcase.
'''
results = search_system.search_music(query, top_k=200)

# play music
# print("\nOverlapping Music：")
# for audio_path in results['audio_paths']:
#     print(f"\nNow playing: {os.path.basename(audio_path)}")
#     display(Audio(audio_path))

# Show explanation of LLM
df_recommendations = pd.DataFrame(results["final_results"])
df_recommendations

# print("\nLLM explanation：")
# print(results['explanation'])

,song_name,artist,description,similarity_score,similarity_audio,similarity_text,sorce,audio_path
0,The Happy Intro,Korolkov,"A modern, upbeat and stylish electronic track ...",0.965010,0.913424,0.987118,"audio,text",music/Korolkov - The Happy Intro.mp3
1,Fashion House Loop,Sawtooz,A powerful and energetic house track with a ca...,0.963338,0.877793,1.000000,"audio,text",music/Sawtooz - Fashion House Loop.mp3
2,Disco Guitar Groove,Frontmusic,"A groovy, funky, and upbeat disco track with a...",0.889935,0.929844,0.872831,"audio,text",music/Frontmusic - Disco Guitar Groove.mp3
3,Flow Free - Short Version,Manos Mars,"A dreamy, reflective, and introspective indie ...",0.877743,0.925198,0.857405,"audio,text",music/Manos Mars - Flow Free - Short Version.mp3
4,Event Ceremony Logo,Artlist Musical Logos,"This is a powerful, energetic and dynamic roya...",0.830021,0.865988,0.814607,"audio,text",music/Artlist Musical Logos - Event Ceremony L...
...,...,...,...,...,...,...,...,...
195,Wild West Slow Dobro Guitar,ilovemedia-es,This short musical cue features a Spanish guit...,0.282382,0.550465,0.167489,"audio,text",music/ilovemedia-es - Wild West Slow Dobro Gui...
196,King of Mandol - Short Version B,LMOP,"A traditional Irish folk track with a upbeat, ...",0.267353,0.817811,0.031442,"audio,text",music/LMOP - King of Mandol - Short Version B.mp3
197,Sad Piano,PineAppleMusic,"A subtle, delicate and thoughtful piano piece ...",0.248572,0.649160,0.076891,"audio,text",music/PineAppleMusic - Sad Piano.wav
198,Little Ragtime,DariusMusicProduction,"A vintage, retro, 1920's style ragtime piano t...",0.195707,0.652356,0.000000,"audio,text",music/DariusMusicProduction - Little Ragtime.mp3


In [87]:
df_recommendations.sort_values("similarity_audio", ascending=False).head(10)

,song_name,artist,description,similarity_score,similarity_audio,similarity_text,sorce,audio_path
39,Childrens Loop,plastic3,This upbeat electronic track features a warm p...,0.631474,1.000000,0.473534,"audio,text",music/plastic3 - Childrens Loop.wav
24,Psy Trance Progressive,Eugene_Barduja,A high energy techno loop with a strong bassli...,0.655908,0.995473,0.510379,"audio,text",music/Eugene_Barduja - Psy Trance Progressive.wav
86,For Kids,Eugene_Barduja,"A happy and cheerful track with ukulele, whist...",0.523838,0.969167,0.332982,"audio,text",music/Eugene_Barduja - For Kids.wav
20,The Truth Is Close,Artlist Musical Logos,A dark and suspenseful piece of electronic mus...,0.666562,0.961149,0.540310,"audio,text",music/Artlist Musical Logos - The Truth Is Clo...
5,Positive Persistent Pluck Sequence 2,Artlist Musical Logos,A groovy and energetic royalty free electronic...,0.784451,0.932071,0.721185,"audio,text",music/Artlist Musical Logos - Positive Persist...
2,Disco Guitar Groove,Frontmusic,"A groovy, funky, and upbeat disco track with a...",0.889935,0.929844,0.872831,"audio,text",music/Frontmusic - Disco Guitar Groove.mp3
3,Flow Free - Short Version,Manos Mars,"A dreamy, reflective, and introspective indie ...",0.877743,0.925198,0.857405,"audio,text",music/Manos Mars - Flow Free - Short Version.mp3
160,Happy Halloween 4,Anandavana,"A fast-paced, playful piece with a mischievous...",0.409248,0.918041,0.191193,"audio,text",music/Anandavana - Happy Halloween 4.mp3
118,Funny Game Loop,honey_lemon,"A casual, jazzy, swing music with vibraphone, ...",0.477178,0.915885,0.289161,"audio,text",music/honey_lemon - Funny Game Loop.wav
0,The Happy Intro,Korolkov,"A modern, upbeat and stylish electronic track ...",0.965010,0.913424,0.987118,"audio,text",music/Korolkov - The Happy Intro.mp3


In [88]:
df_recommendations.sort_values("similarity_text", ascending=False).head(10)

,song_name,artist,description,similarity_score,similarity_audio,similarity_text,sorce,audio_path
1,Fashion House Loop,Sawtooz,A powerful and energetic house track with a ca...,0.963338,0.877793,1.000000,"audio,text",music/Sawtooz - Fashion House Loop.mp3
0,The Happy Intro,Korolkov,"A modern, upbeat and stylish electronic track ...",0.965010,0.913424,0.987118,"audio,text",music/Korolkov - The Happy Intro.mp3
2,Disco Guitar Groove,Frontmusic,"A groovy, funky, and upbeat disco track with a...",0.889935,0.929844,0.872831,"audio,text",music/Frontmusic - Disco Guitar Groove.mp3
3,Flow Free - Short Version,Manos Mars,"A dreamy, reflective, and introspective indie ...",0.877743,0.925198,0.857405,"audio,text",music/Manos Mars - Flow Free - Short Version.mp3
4,Event Ceremony Logo,Artlist Musical Logos,"This is a powerful, energetic and dynamic roya...",0.830021,0.865988,0.814607,"audio,text",music/Artlist Musical Logos - Event Ceremony L...
7,Energy Dance Loop,Difourks,A powerful and energetic electro track with du...,0.760850,0.639460,0.812874,"audio,text",music/Difourks - Energy Dance Loop.wav
15,Energetic Loop,Artlist Musical Logos,"A hard-hitting, edgy, gritty, dark and driving...",0.692985,0.612319,0.727556,"audio,text",music/Artlist Musical Logos - Energetic Loop.mp3
9,The 8 Oclock Story - Intro,Ziv Moran,This is a bright and uplifting background musi...,0.757281,0.840413,0.721653,"audio,text",music/Ziv Moran - The 8 Oclock Story - Intro.mp3
5,Positive Persistent Pluck Sequence 2,Artlist Musical Logos,A groovy and energetic royalty free electronic...,0.784451,0.932071,0.721185,"audio,text",music/Artlist Musical Logos - Positive Persist...
6,Fortitude - Short Version,Lance Conrad,A cinematic track that builds from a minimal p...,0.766017,0.879387,0.717429,"audio,text",music/Lance Conrad - Fortitude - Short Version...


In [9]:
def precision_at_k(gt, recs, k):
    recs_at_k = recs[:k]
    relevant = [r for r in recs_at_k if r in gt]
    return len(relevant) / k

def recall_at_k(gt, recs, k):
    recs_at_k = recs[:k]
    relevant = [r for r in recs_at_k if r in gt]
    return len(relevant) / len(gt) if len(gt) > 0 else 0.0

def ndcg_at_k(gt, recs, k):
    recs_at_k = recs[:k]
    dcg = 0.0
    for i, rec in enumerate(recs_at_k):
        if rec in gt:
            dcg += 1 / np.log2(i + 2)  # log2(i+2) since i starts from 0
    ideal_rels = [1] * min(len(gt), k)
    idcg = sum([rel / np.log2(i + 2) for i, rel in enumerate(ideal_rels)])
    return dcg / idcg if idcg > 0 else 0.0

In [89]:
import numpy as np

gt = [137, 148, 158]
b_recommended = [106, 137, 158]
a_recommended = [110, 190, 107]
t_recommended = [137, 106, 158]


methods = ['B', 'A', 'T']
recommendations = [b_recommended, a_recommended, t_recommended]

results = []
for method, rec in zip(methods, recommendations):
    results.append({
        'Method': method,
        'Precision@5': precision_at_k(gt, rec, 3),
        'Recall@5': recall_at_k(gt, rec, 3),
        'nDCG@5': ndcg_at_k(gt, rec, 3)
    })

df = pd.DataFrame(results)
df

,Method,Precision@5,Recall@5,nDCG@5
0,B,0.666667,0.666667,0.530721
1,A,0.000000,0.000000,0.000000
2,T,0.666667,0.666667,0.703918
